# Week 13 — Validation Protocol

## Objective

Design a reproducible validation protocol for
spatiotemporal road accident risk prediction.

The main objectives are:

- Chronological train, validation and final test separation.
- Leakage-safe historical feature construction.
- Nested temporal cross-validation.
- Hyperparameter tuning within training folds.
- Grouped validation by intersection ID.
- Evaluation using Average Precision.
- Preservation of an untouched final test period.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    TimeSeriesSplit,
    GroupKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

In [2]:
df = pd.read_csv(
    "../data/processed/intersection_day_2018_features.csv",
    parse_dates=["prediction_date", "target_date"]
)

df = df.sort_values(
    ["prediction_date", "intersection_id"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (823217, 16)
Columns: ['intersection_id', 'prediction_date', 'target_date', 'historical_ksi_count', 'historical_traffic_volume', 'day_of_week', 'month', 'target', 'ksi_count_30d', 'ksi_count_365d', 'is_weekend', 'traffic_missing', 'weather_date', 'temperature_prev_day', 'precipitation_prev_day', 'weather_missing']


In [3]:
RANDOM_STATE = 42
PREDICTION_HORIZON = 1
TEMPORAL_GAP = 1
GROUP_COLUMN = "intersection_id"

TRAIN_START = pd.Timestamp("2018-01-01")
TRAIN_END = pd.Timestamp("2018-09-29")

VAL_START = pd.Timestamp("2018-10-01")
VAL_END = pd.Timestamp("2018-12-31")

TEST_START = pd.Timestamp("2019-01-02")
TEST_END = pd.Timestamp("2019-12-30")

In [4]:
train_mask = df["prediction_date"].between(
    TRAIN_START,
    TRAIN_END
)

val_mask = df["prediction_date"].between(
    VAL_START,
    VAL_END
)

train_df = df.loc[train_mask].copy().reset_index(drop=True)
val_df = df.loc[val_mask].copy().reset_index(drop=True)

In [5]:
split_summary = pd.DataFrame({
    "split": ["train", "validation"],
    "start_date": [
        train_df["prediction_date"].min(),
        val_df["prediction_date"].min()
    ],
    "end_date": [
        train_df["prediction_date"].max(),
        val_df["prediction_date"].max()
    ],
    "rows": [
        len(train_df),
        len(val_df)
    ],
    "positives": [
        train_df["target"].sum(),
        val_df["target"].sum()
    ]
})

split_summary["positive_rate"] = (
    split_summary["positives"] / split_summary["rows"]
)

assert (
    train_df["target_date"].max()
    < val_df["prediction_date"].min()
)

assert (
    val_df["target_date"].max()
    < TEST_START
)

split_summary

,split,start_date,end_date,rows,positives,positive_rate
0,train,2018-01-01,2018-09-29,613223,81,0.000132
1,validation,2018-10-01,2018-12-31,207736,38,0.000183


In [6]:
events = pd.read_csv(
    "../data/processed/ksi_matched_events.csv",
    usecols=["intersection_id", "collision_id", "event_date"],
    parse_dates=["event_date"]
)

events = events.loc[
    events["event_date"] < VAL_END
].copy()

event_history = {
    intersection_id: np.sort(
        group["event_date"].to_numpy(dtype="datetime64[D]")
    )
    for intersection_id, group in events.groupby("intersection_id")
}

In [7]:
history_columns = [
    "historical_ksi_count",
    "ksi_count_30d",
    "ksi_count_365d"
]

feature_changes = []

for split_name, period_df in [
    ("train", train_df),
    ("validation", val_df)
]:
    prediction_dates = period_df[
        "prediction_date"
    ].to_numpy(dtype="datetime64[D]")

    counts = np.zeros(
        (len(period_df), len(history_columns)),
        dtype=np.int64
    )

    for intersection_id, positions in period_df.groupby(
        "intersection_id"
    ).indices.items():

        dates = prediction_dates[positions]
        history = event_history.get(intersection_id)

        if history is None:
            continue

        right = np.searchsorted(
            history, dates, side="left"
        )

        left_30 = np.searchsorted(
            history,
            dates - np.timedelta64(30, "D"),
            side="left"
        )

        left_365 = np.searchsorted(
            history,
            dates - np.timedelta64(365, "D"),
            side="left"
        )

        counts[positions, 0] = right
        counts[positions, 1] = right - left_30
        counts[positions, 2] = right - left_365

    changes = (
        period_df[history_columns].to_numpy() != counts
    ).sum(axis=0)

    feature_changes.append({
        "split": split_name,
        **dict(zip(history_columns, changes))
    })

    period_df[history_columns] = counts

In [8]:
sample = val_df.loc[
    val_df["ksi_count_30d"].idxmax()
]

sample_date = sample["prediction_date"]

past_events = events.loc[
    (events["intersection_id"] == sample["intersection_id"])
    & (events["event_date"] < sample_date)
]

expected = np.array([
    len(past_events),
    (
        past_events["event_date"]
        >= sample_date - pd.Timedelta(days=30)
    ).sum(),
    (
        past_events["event_date"]
        >= sample_date - pd.Timedelta(days=365)
    ).sum()
])

actual = sample[history_columns].to_numpy(dtype=np.int64)

assert np.array_equal(expected, actual)

pd.DataFrame(feature_changes)

,split,historical_ksi_count,ksi_count_30d,ksi_count_365d
0,train,0,0,0
1,validation,0,0,0


In [9]:
OUTER_N_SPLITS = 3

train_dates = np.sort(
    train_df["prediction_date"].unique()
)

outer_cv = TimeSeriesSplit(
    n_splits=OUTER_N_SPLITS,
    gap=TEMPORAL_GAP
)

print("Unique training dates:", len(train_dates))

Unique training dates: 272


In [10]:
outer_splits = []
outer_fold_summary = []

for fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(train_dates),
    start=1
):
    fold_train_dates = train_dates[train_idx]
    fold_val_dates = train_dates[val_idx]

    train_positions = np.flatnonzero(
        train_df["prediction_date"].isin(fold_train_dates)
    )

    val_positions = np.flatnonzero(
        train_df["prediction_date"].isin(fold_val_dates)
    )

    fold_train = train_df.iloc[train_positions]
    fold_val = train_df.iloc[val_positions]

    outer_splits.append(
        (train_positions, val_positions)
    )

    outer_fold_summary.append({
        "fold": fold,
        "train_start": fold_train_dates.min(),
        "train_end": fold_train_dates.max(),
        "val_start": fold_val_dates.min(),
        "val_end": fold_val_dates.max(),
        "train_rows": len(fold_train),
        "val_rows": len(fold_val),
        "train_positives": int(fold_train["target"].sum()),
        "val_positives": int(fold_val["target"].sum())
    })

In [11]:
outer_fold_summary = pd.DataFrame(
    outer_fold_summary
)

for train_idx, val_idx in outer_splits:
    fold_train = train_df.iloc[train_idx]
    fold_val = train_df.iloc[val_idx]

    assert (
        fold_train["target_date"].max()
        < fold_val["prediction_date"].min()
    )

    assert fold_train["target"].nunique() == 2
    assert fold_val["target"].nunique() == 2

outer_fold_summary

,fold,train_start,train_end,val_start,val_end,train_rows,val_rows,train_positives,val_positives
0,1,2018-01-01,2018-03-08,2018-03-10,2018-05-16,150929,153204,14,20
1,2,2018-01-01,2018-05-15,2018-05-17,2018-07-23,304133,153315,34,18
2,3,2018-01-01,2018-07-22,2018-07-24,2018-09-29,457444,153522,51,29


In [12]:
INNER_N_SPLITS = 2

inner_cv = TimeSeriesSplit(
    n_splits=INNER_N_SPLITS,
    gap=TEMPORAL_GAP
)

inner_splits_by_outer = {}
inner_fold_summary = []

for outer_fold, (outer_train_idx, _) in enumerate(
    outer_splits,
    start=1
):
    outer_train_df = train_df.iloc[
        outer_train_idx
    ].reset_index(drop=True)

    outer_train_dates = np.sort(
        outer_train_df["prediction_date"].unique()
    )

    inner_splits = []

    for inner_fold, (train_idx, val_idx) in enumerate(
        inner_cv.split(outer_train_dates),
        start=1
    ):
        inner_train_dates = outer_train_dates[train_idx]
        inner_val_dates = outer_train_dates[val_idx]

        inner_train_positions = np.flatnonzero(
            outer_train_df["prediction_date"].isin(
                inner_train_dates
            )
        )

        inner_val_positions = np.flatnonzero(
            outer_train_df["prediction_date"].isin(
                inner_val_dates
            )
        )

        inner_train = outer_train_df.iloc[
            inner_train_positions
        ]

        inner_val = outer_train_df.iloc[
            inner_val_positions
        ]

        inner_splits.append(
            (inner_train_positions, inner_val_positions)
        )

        inner_fold_summary.append({
            "outer_fold": outer_fold,
            "inner_fold": inner_fold,
            "train_start": inner_train_dates.min(),
            "train_end": inner_train_dates.max(),
            "val_start": inner_val_dates.min(),
            "val_end": inner_val_dates.max(),
            "train_rows": len(inner_train),
            "val_rows": len(inner_val),
            "train_positives": int(inner_train["target"].sum()),
            "val_positives": int(inner_val["target"].sum())
        })

    inner_splits_by_outer[outer_fold] = inner_splits

In [13]:
inner_fold_summary = pd.DataFrame(
    inner_fold_summary
)

assert (
    inner_fold_summary["train_end"]
    + pd.Timedelta(days=PREDICTION_HORIZON)
    < inner_fold_summary["val_start"]
).all(), "Temporal leakage detected"

for outer_fold, (outer_train_idx, _) in enumerate(
    outer_splits,
    start=1
):
    outer_y = train_df.iloc[outer_train_idx]["target"]

    for inner_train_idx, inner_val_idx in inner_splits_by_outer[
        outer_fold
    ]:
        assert outer_y.iloc[inner_train_idx].nunique() == 2, (
            f"Missing class in outer fold {outer_fold} inner train"
        )

        assert outer_y.iloc[inner_val_idx].nunique() == 2, (
            f"Missing class in outer fold {outer_fold} inner validation"
        )

inner_fold_summary

,outer_fold,inner_fold,train_start,train_end,val_start,val_end,train_rows,val_rows,train_positives,val_positives
0,1,1,2018-01-01,2018-01-22,2018-01-24,2018-02-14,49544,49566,2,6
1,1,2,2018-01-01,2018-02-13,2018-02-15,2018-03-08,99110,49566,9,5
2,2,1,2018-01-01,2018-02-13,2018-02-15,2018-03-31,99110,101385,9,9
3,2,2,2018-01-01,2018-03-30,2018-04-01,2018-05-15,200495,101385,18,16
4,3,1,2018-01-01,2018-03-09,2018-03-11,2018-05-16,153182,150951,14,20
5,3,2,2018-01-01,2018-05-15,2018-05-17,2018-07-22,304133,151058,34,17


In [14]:
numeric_features = [
    "historical_ksi_count",
    "historical_traffic_volume",
    "ksi_count_30d",
    "ksi_count_365d",
    "temperature_prev_day",
    "precipitation_prev_day"
]

categorical_features = [
    "day_of_week",
    "month"
]

binary_features = [
    "is_weekend",
    "traffic_missing",
    "weather_missing"
]

features = (
    numeric_features
    + categorical_features
    + binary_features
)

X_train = train_df[features]
y_train = train_df["target"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
    ("binary", "passthrough", binary_features)
])

In [15]:
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

nested_results = []

for outer_fold, (train_idx, val_idx) in enumerate(
    outer_splits,
    start=1
):
    X_outer_train = X_train.iloc[train_idx]
    y_outer_train = y_train.iloc[train_idx]

    X_outer_val = X_train.iloc[val_idx]
    y_outer_val = y_train.iloc[val_idx]

    search = GridSearchCV(
        estimator=model_pipeline,
        param_grid=param_grid,
        cv=inner_splits_by_outer[outer_fold],
        scoring="average_precision",
        refit=True,
        n_jobs=1,
        error_score="raise"
    )

    search.fit(X_outer_train, y_outer_train)

    y_proba = search.best_estimator_.predict_proba(
        X_outer_val
    )[:, 1]

    outer_ap = average_precision_score(
        y_outer_val,
        y_proba
    )

    nested_results.append({
        "outer_fold": outer_fold,
        "best_C": search.best_params_["model__C"],
        "inner_cv_ap": search.best_score_,
        "outer_ap": outer_ap,
        "baseline_ap": y_outer_val.mean(),
        "train_positives": int(y_outer_train.sum()),
        "val_positives": int(y_outer_val.sum())
    })

    print(
        f"Outer fold {outer_fold} completed | "
        f"C={search.best_params_['model__C']} | "
        f"Outer AP={outer_ap:.6f}"
    )

Outer fold 1 completed | C=0.01 | Outer AP=0.000150
Outer fold 2 completed | C=0.01 | Outer AP=0.000399
Outer fold 3 completed | C=0.1 | Outer AP=0.002162


In [16]:
nested_results = pd.DataFrame(nested_results)

nested_mean_ap = nested_results["outer_ap"].mean()

nested_std_ap = nested_results["outer_ap"].std(ddof=0)

nested_results["ap_lift"] = (
    nested_results["outer_ap"]
    / nested_results["baseline_ap"]
)

print(f"Nested CV mean AP: {nested_mean_ap:.6f}")
print(f"Nested CV std AP:  {nested_std_ap:.6f}")

nested_results.round(6)

Nested CV mean AP: 0.000904
Nested CV std AP:  0.000895


,outer_fold,best_C,inner_cv_ap,outer_ap,baseline_ap,train_positives,val_positives,ap_lift
0,1,0.01,0.000351,0.000150,0.000131,14,20,1.150917
1,2,0.01,0.000262,0.000399,0.000117,34,18,3.398137
2,3,0.10,0.000399,0.002162,0.000189,51,29,11.444112


In [17]:
from sklearn.base import clone

GROUP_N_SPLITS = 3
GROUP_C = 0.1

groups = train_df[GROUP_COLUMN]

group_cv = GroupKFold(
    n_splits=GROUP_N_SPLITS
)

group_model = clone(model_pipeline)

group_model.set_params(
    model__C=GROUP_C
)

group_results = []
group_assignments = []

In [18]:
for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X_train, y_train, groups=groups),
    start=1
):
    X_group_train = X_train.iloc[train_idx]
    y_group_train = y_train.iloc[train_idx]

    X_group_val = X_train.iloc[val_idx]
    y_group_val = y_train.iloc[val_idx]

    train_groups = groups.iloc[train_idx]
    val_groups = groups.iloc[val_idx]

    assert not set(train_groups).intersection(val_groups)

    assert y_group_train.nunique() == 2
    assert y_group_val.nunique() == 2

    estimator = clone(group_model)

    estimator.fit(
        X_group_train,
        y_group_train
    )

    y_proba = estimator.predict_proba(
        X_group_val
    )[:, 1]

    group_ap = average_precision_score(
        y_group_val,
        y_proba
    )

    group_results.append({
        "fold": fold,
        "train_rows": len(X_group_train),
        "val_rows": len(X_group_val),
        "train_groups": train_groups.nunique(),
        "val_groups": val_groups.nunique(),
        "train_positives": int(y_group_train.sum()),
        "val_positives": int(y_group_val.sum()),
        "average_precision": group_ap,
        "baseline_ap": y_group_val.mean()
    })

    group_assignments.extend([
        {
            "intersection_id": intersection_id,
            "validation_fold": fold
        }
        for intersection_id in val_groups.unique()
    ])

    print(
        f"Group fold {fold} completed | "
        f"AP={group_ap:.6f}"
    )

Group fold 1 completed | AP=0.000182
Group fold 2 completed | AP=0.000222
Group fold 3 completed | AP=0.004224


In [19]:
group_results = pd.DataFrame(group_results)

group_assignments = pd.DataFrame(
    group_assignments
).sort_values("intersection_id").reset_index(drop=True)

group_mean_ap = group_results[
    "average_precision"
].mean()

group_std_ap = group_results[
    "average_precision"
].std(ddof=0)

group_results["ap_lift"] = (
    group_results["average_precision"]
    / group_results["baseline_ap"]
)

print(f"Grouped CV mean AP: {group_mean_ap:.6f}")
print(f"Grouped CV std AP:  {group_std_ap:.6f}")

group_results.round(6)

Grouped CV mean AP: 0.001542
Grouped CV std AP:  0.001896


,fold,train_rows,val_rows,train_groups,val_groups,train_positives,val_positives,average_precision,baseline_ap,ap_lift
0,1,408848,204375,1506,752,58,23,0.000182,0.000113,1.613778
1,2,408781,204442,1505,753,49,32,0.000222,0.000157,1.416522
2,3,408817,204406,1505,753,55,26,0.004224,0.000127,33.204919


In [20]:
final_cv = TimeSeriesSplit(
    n_splits=INNER_N_SPLITS,
    gap=TEMPORAL_GAP
)

final_splits = []
final_fold_summary = []

for fold, (train_idx, val_idx) in enumerate(
    final_cv.split(train_dates),
    start=1
):
    fold_train_dates = train_dates[train_idx]
    fold_val_dates = train_dates[val_idx]

    train_positions = np.flatnonzero(
        train_df["prediction_date"].isin(fold_train_dates)
    )

    val_positions = np.flatnonzero(
        train_df["prediction_date"].isin(fold_val_dates)
    )

    fold_train = train_df.iloc[train_positions]
    fold_val = train_df.iloc[val_positions]

    assert (
        fold_train["target_date"].max()
        < fold_val["prediction_date"].min()
    )

    assert fold_train["target"].nunique() == 2
    assert fold_val["target"].nunique() == 2

    final_splits.append(
        (train_positions, val_positions)
    )

    final_fold_summary.append({
        "fold": fold,
        "train_start": fold_train_dates.min(),
        "train_end": fold_train_dates.max(),
        "val_start": fold_val_dates.min(),
        "val_end": fold_val_dates.max(),
        "train_rows": len(fold_train),
        "val_rows": len(fold_val),
        "train_positives": int(fold_train["target"].sum()),
        "val_positives": int(fold_val["target"].sum())
    })

final_fold_summary = pd.DataFrame(final_fold_summary)

final_fold_summary

,fold,train_start,train_end,val_start,val_end,train_rows,val_rows,train_positives,val_positives
0,1,2018-01-01,2018-04-01,2018-04-03,2018-07-01,205001,202803,18,28
1,2,2018-01-01,2018-06-30,2018-07-02,2018-09-29,407802,203166,46,35


In [21]:
final_search = GridSearchCV(
    estimator=clone(model_pipeline),
    param_grid=param_grid,
    cv=final_splits,
    scoring="average_precision",
    refit=True,
    n_jobs=1,
    error_score="raise"
)

final_search.fit(X_train, y_train)

selected_model = final_search.best_estimator_
selected_C = final_search.best_params_["model__C"]

print(f"Selected C: {selected_C}")
print(f"Final tuning CV AP: {final_search.best_score_:.6f}")

Selected C: 100
Final tuning CV AP: 0.000882


In [22]:
X_val = val_df[features]
y_val = val_df["target"]

val_proba = selected_model.predict_proba(X_val)[:, 1]

development_ap = average_precision_score(
    y_val,
    val_proba
)

development_baseline_ap = y_val.mean()

development_lift = (
    development_ap / development_baseline_ap
)

print(f"Selected C: {selected_C}")
print(f"Development AP: {development_ap:.6f}")
print(f"Constant baseline AP: {development_baseline_ap:.6f}")
print(f"Development AP lift: {development_lift:.2f}x")

Selected C: 100
Development AP: 0.000390
Constant baseline AP: 0.000183
Development AP lift: 2.13x


In [23]:
historical_rates = train_df.groupby(
    GROUP_COLUMN
)["target"].mean()

global_rate = y_train.mean()

historical_val_scores = (
    val_df[GROUP_COLUMN]
    .map(historical_rates)
    .fillna(global_rate)
)

historical_development_ap = average_precision_score(
    y_val,
    historical_val_scores
)

print(
    f"Historical baseline AP: "
    f"{historical_development_ap:.6f}"
)

Historical baseline AP: 0.000182


In [24]:
validation_comparison = pd.DataFrame([
    {
        "protocol": "Standard Temporal CV (W12)",
        "mean_ap": 0.000976,
        "std_ap": np.nan,
        "folds": 3,
        "parameter_selection": "Previous W12 tuning"
    },
    {
        "protocol": "Nested Temporal CV (W13)",
        "mean_ap": nested_mean_ap,
        "std_ap": nested_std_ap,
        "folds": OUTER_N_SPLITS,
        "parameter_selection": "Nested grid search"
    },
    {
        "protocol": "Grouped CV (W13)",
        "mean_ap": group_mean_ap,
        "std_ap": group_std_ap,
        "folds": GROUP_N_SPLITS,
        "parameter_selection": "Fixed C=0.1"
    }
])

validation_comparison.round(6)

,protocol,mean_ap,std_ap,folds,parameter_selection
0,Standard Temporal CV (W12),0.000976,NaN,3,Previous W12 tuning
1,Nested Temporal CV (W13),0.000904,0.000895,3,Nested grid search
2,Grouped CV (W13),0.001542,0.001896,3,Fixed C=0.1


In [25]:
development_comparison = pd.DataFrame([
    {
        "model": "Constant baseline",
        "average_precision": development_baseline_ap
    },
    {
        "model": "Historical frequency baseline",
        "average_precision": historical_development_ap
    },
    {
        "model": "Logistic Regression",
        "average_precision": development_ap
    }
])

development_comparison["ap_lift"] = (
    development_comparison["average_precision"]
    / development_baseline_ap
)

development_comparison.round(6)

,model,average_precision,ap_lift
0,Constant baseline,0.000183,1.000000
1,Historical frequency baseline,0.000182,0.993478
2,Logistic Regression,0.000390,2.130863


In [26]:
from pathlib import Path

REPORT_DIR = Path("../reports/validation_w13")

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert group_assignments["intersection_id"].is_unique
assert set(group_assignments["intersection_id"]) == set(groups)

report_tables = {
    "split_summary": split_summary,
    "historical_feature_changes": pd.DataFrame(feature_changes),
    "outer_folds": outer_fold_summary,
    "inner_folds": inner_fold_summary,
    "final_tuning_folds": final_fold_summary,
    "group_assignments": group_assignments,
    "nested_results": nested_results,
    "group_results": group_results,
    "validation_comparison": validation_comparison,
    "development_comparison": development_comparison
}

for name, table in report_tables.items():
    table.to_csv(
        REPORT_DIR / f"{name}.csv",
        index=False
    )

print(f"Saved {len(report_tables)} report tables")

Saved 10 report tables


In [27]:
import json
import platform
import sklearn

validation_protocol = {
    "project": "Road Accident Risk Intelligence",
    "week": 13,
    "dataset": "intersection_day_2018_features.csv",
    "observation_unit": "intersection_day",
    "target": "next_day_matched_ksi_event",
    "prediction_horizon_days": PREDICTION_HORIZON,
    "temporal_gap_days": TEMPORAL_GAP,
    "group_column": GROUP_COLUMN,
    "random_state": RANDOM_STATE,
    "periods": {
        "train": {
            "start": TRAIN_START.strftime("%Y-%m-%d"),
            "end": TRAIN_END.strftime("%Y-%m-%d")
        },
        "development_validation": {
            "start": VAL_START.strftime("%Y-%m-%d"),
            "end": VAL_END.strftime("%Y-%m-%d")
        },
        "final_test": {
            "start": TEST_START.strftime("%Y-%m-%d"),
            "end": TEST_END.strftime("%Y-%m-%d"),
            "dataset_built": False,
            "evaluated": False
        }
    },
    "historical_features": {
        "source": "ksi_matched_events.csv",
        "method": "numpy.searchsorted",
        "rolling_windows_days": [30, 365],
        "include_prediction_day": False,
        "reporting_delay_verified": False
    },
    "features": {
        "numeric": numeric_features,
        "categorical": categorical_features,
        "binary": binary_features
    },
    "cross_validation": {
        "temporal_splitter": "TimeSeriesSplit",
        "temporal_split_unit": "prediction_date",
        "outer_folds": OUTER_N_SPLITS,
        "inner_folds": INNER_N_SPLITS,
        "final_tuning_folds": INNER_N_SPLITS,
        "group_splitter": "GroupKFold",
        "group_folds": GROUP_N_SPLITS
    },
    "model": {
        "name": "LogisticRegression",
        "max_iter": 1000,
        "parameter_grid": param_grid,
        "grouped_cv_C": GROUP_C,
        "selected_C": float(selected_C)
    },
    "metric": "average_precision",
    "results": {
        "nested_mean_ap": float(nested_mean_ap),
        "nested_std_ap": float(nested_std_ap),
        "grouped_mean_ap": float(group_mean_ap),
        "grouped_std_ap": float(group_std_ap),
        "final_tuning_ap": float(final_search.best_score_),
        "development_ap": float(development_ap),
        "constant_baseline_ap": float(development_baseline_ap),
        "historical_baseline_ap": float(historical_development_ap),
        "development_ap_lift": float(development_lift)
    },
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__
    }
}

with open(
    REPORT_DIR / "validation_protocol.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        validation_protocol,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Validation protocol saved successfully")

Validation protocol saved successfully


## 9. Conclusions

### Key Results

- Nested temporal cross-validation achieved a mean Average Precision of 0.000904 with a standard deviation of 0.000895.
- Grouped cross-validation achieved a mean Average Precision of 0.001542 with a standard deviation of 0.001896.
- Final temporal hyperparameter tuning selected C = 100.
- Logistic Regression achieved a development Average Precision of 0.000390.
- The constant and historical frequency baselines achieved Average Precision values of 0.000183 and 0.000182, respectively.
- Logistic Regression achieved a 2.13x AP lift over the constant baseline.

### Methodological Achievements

- Chronological development periods were preserved.
- Temporal folds were constructed using unique prediction dates.
- Historical KSI features were independently reconstructed using strictly preceding event dates.
- Hyperparameter selection was separated from outer model evaluation.
- Preprocessing was fitted exclusively within the corresponding training folds.
- Intersection-level generalization was evaluated using GroupKFold.
- Temporal boundaries, group assignments and experiment results were exported.

### Limitations

- Positive KSI observations remain extremely rare.
- The smallest inner training fold contains only two positive observations.
- Temporal and grouped evaluation results exhibit substantial variability.
- Hyperparameter selection varies across training periods.
- Grouped validation does not enforce chronological separation.
- Historical event reporting delays have not been verified.
- Development data has been reused across multiple project weeks.
- The reserved 2019 final test dataset has not been constructed or evaluated.
- Probability calibration and practical decision utility have not yet been established.

### Conclusion

Week 13 establishes a reproducible validation protocol for road accident risk modelling. Logistic Regression demonstrates ranking performance above simple baselines, but its stability and practical generalization remain limited.

An independent chronological final test is required before drawing definitive conclusions about model performance.